In [ ]:
"""
https://www.bambooweekly.com/bw-4-eating-well/
https://www.bambooweekly.com/bw-4-eating-well-solution/
https://github.com/JoergEm/Bamboo-Weekly/tree/main 
"""

In [ ]:
from IPython.display import FileLink, Markdown
import matplotlib.pyplot as plt 
import numpy as np
import os
import pandas as pd
from pathlib import Path
import requests
import shutil
import tempfile
display(Markdown("Imports ✅"))

In [ ]:
try:
    import patoolib
    import sas7bdat
except ImportError:
    !pip install patool
    !pip install sas7bdat
    import patoolib
    import sas7bdat
finally:
    display(Markdown("Installs ✅"))

In [ ]:
def create_folders(folders: list[str]) -> bool:
    try:
        for folder in folders:
            folderpath: str = os.path.join(os.getcwd(), folder)
            if not os.path.exists(folderpath):
                os.makedirs(folderpath, exist_ok=True)
    except:
        print("Folder {folderpath} could not be created.")
        return False
    else:
        display(Markdown("Folders ✅"))
        return True

In [ ]:
def download_data(url: str, filename: str) -> bool:
    from urllib.request import urlretrieve
    from urllib.error import HTTPError
    try:
        urlretrieve(url, filename)
        return True
    except HTTPError as e:
        if e.code == 403:
            import requests
            try:
                response: requests.Response = requests.get(url)
                with open(filename, 'wb') as f:
                    f.write(response.content)
                    return True
            except:
                print("Could not download Data")
                return False
    return False

In [ ]:
url: str = 'https://www2.census.gov/programs-surveys/nsch/datasets/2021/nsch_2021_topical_SAS.zip'
extracted_folder: str = os.path.join('data', 'nsch_2021_topical_SAS')
folders: list[str] = ['data', 'results', extracted_folder]
filename: str = 'nsch_2021e_topical.sas7bdat'
filepath: str = os.path.join(folders[0], 'nsch_2021_topical_SAS', filename)
archivename: str = 'nsch_2021_topical_SAS.zip'
archivepath: str = os.path.join(folders[0], archivename)
create_folders(folders)

if not os.path.exists(filepath):
    if download_data(url, archivepath):
        if patoolib.extract_archive(archivepath, outdir=extracted_folder):
            print("Contents of extracted folder:")
            extracted_files: str = os.listdir(extracted_folder)
            print(extracted_files)  
            sas_file: str | None = None
            for file in extracted_files:
                  if file.endswith('.sas7bdat'):
                    sas_file= os.path.join(extracted_folder, file)
                    break
            if sas_file and os.path.exists(sas_file): 
                try:
                    data: pd.DataFrame = pd.read_sas(sas_file, format='sas7bdat', encoding="ISO-8859-1")
                    display(Markdown("Data ✅"))
                except Exception as e:
                    print("Could not read Data")
            else:
                print("No valid SAS file found.")
    else:
        display(Markdown('Error ❌'))
else:
    data: pd.DataFrame = pd.read_sas(filepath, format='sas7bdat', encoding="ISO-8859-1")
    display(Markdown("Data loaded from existing file ✅"))  

if os.path.exists(filepath):
    display(FileLink(filepath))

In [ ]:
# Download the topical data file, in SAS format, from: https://www2.census.gov/programs-surveys/nsch/datasets/2021/nsch_2021_topical_SAS.zip. Turn it into a data frame. We're only interested in the following columns:    FIPSST    VEGETABLE    FRUIT    SUGARDRINK
data = data[['FIPSST', 'VEGETABLES', 'FRUIT', 'SUGARDRINK']]

In [ ]:
# Turn the FIPSST column into an integer, and make it the index.
data['FIPSST'] = data['FIPSST'].astype(np.int8)
data = data.set_index('FIPSST')

In [ ]:
# What percentage of children had, on average, less than one vegetable per day during the week preceding the study?
# data['VEGETABLES'] < 4, 'VEGETABLES' # category to boolean
# data.loc[data['VEGETABLES'] < 4, 'VEGETABLES'] # mask index
# data.loc[data['VEGETABLES'] < 4, 'VEGETABLES'].count() # how many 
data.loc[data['VEGETABLES'] < 4, 'VEGETABLES'].count() / data['VEGETABLES'].count() # ratio

In [ ]:
# What percentage of children had, on average, less than one vegetable per day and less than one fruit per day during the week preceding the study?
# (data['VEGETABLES'] < 4) & (data['FRUIT'] < 4) # category to boolean
# data.loc[(data['VEGETABLES'] < 4) & (data['FRUIT'] < 4)] # mask index
# data.loc[(data['VEGETABLES'] < 4) & (data['FRUIT'] < 4), 'VEGETABLES'] # only vegetables
# data.loc[(data['VEGETABLES'] < 4) & (data['FRUIT'] < 4), 'VEGETABLES'].count() # how many
data.loc[(data['VEGETABLES'] < 4) & (data['FRUIT'] < 4), 'VEGETABLES'].count() / data['VEGETABLES'].count()

In [ ]:
# What percentage of children had, on average, less than one vegetable per day and less than one fruit per day and did have a sugary drink during the week preceding the study?
data.loc[(data['VEGETABLES'] < 4) & (data['FRUIT'] < 4) & (data['SUGARDRINK'] > 1), 'VEGETABLES'].count() / data['VEGETABLES'].count()

In [ ]:
url = 'https://www2.census.gov/geo/docs/reference/state.txt'
folders = ['data', 'results']
filename = 'state.txt'
filepath = os.path.join(folders[0], filename)

if not os.path.exists(filepath):
    if create_folders(folders):
        if download_data(url, filepath):
            try:
                fips_df = pd.read_csv(filepath, sep='|', usecols=['STATE', 'STATE_NAME'], index_col='STATE')
                display(Markdown("Data ✅"))
            except:
                print("Could not read Data")
    else:
        display(Markdown('Error ❌'))
else:
    fips_df = pd.read_csv(filepath, sep='|', usecols=['STATE', 'STATE_NAME'], index_col='STATE')
    display(Markdown("Data loaded from existing file ✅"))  

if os.path.exists(filepath):
    display(FileLink(filepath))

In [ ]:
# Download the FIPS state reference info, in CSV format, from https://www2.census.gov/geo/docs/reference/state.txt. Turn this into a data frame, with the STATE column as the index. The only other column we care about is STATE_NAME.
joined_df = data.join(fips_df)